# 🔬 Incremental Preprocessing & Ablation Pipeline (Stages 1 to 7)

**Project:** Comparative Study of Transfer Learning Models for Acne Severity Classification on ACNE04  
**Author:** Akmal Yaasir Fauzaan  

---

### Objective & Ablation Strategy
In medical image classification, raw images contain unwanted variations such as background artifacts, non-standard illumination, and low contrast around inflammatory lesions.  
This notebook demonstrates an **incremental 7-stage preprocessing protocol** designed to evaluate the isolated and cumulative contribution of each computer vision technique:

1. **Stage 1 (Raw Baseline):** Resize $224 \times 224$ + ImageNet Normalization.
2. **Stage 2 (+ Data Augmentation):** Random Horizontal Flip + Mild Rotation ($15^\circ$) + Color Jitter.
3. **Stage 3 (+ CLAHE):** Contrast Limited Adaptive Histogram Equalization on Luminance ($L$) in LAB space.
4. **Stage 4 (+ Color Constancy):** White Balance correction via Gray World and Shades of Gray algorithms.
5. **Stage 5 (+ Face Cropping):** Automated facial detection with safety margins using OpenCV Haar Cascades.
6. **Stage 6 (+ Skin Masking):** Epidermal segmentation combining HSV and YCbCr color spaces.
7. **Stage 7 (+ Imbalance Handling):** Cost-Sensitive Class Weights ($w_c$) and Random Oversampling (ROS).

> **Note:** This notebook is prepared for exploration and inspection. Model training can be launched whenever ready via CLI.

## 1. Imports & Environment Configuration

In [ ]:
# 1. Environment & Multi-Threading Safety Configuration
%matplotlib inline
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import sys
from pathlib import Path

import cv2
cv2.setNumThreads(0)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch

# Resolve project root dynamically
current_dir = Path(os.getcwd())
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.preprocessing import (
    STAGE_DESCRIPTIONS,
    apply_clahe,
    apply_color_constancy,
    crop_face,
    apply_skin_mask,
    compute_class_weights,
    get_weighted_sampler,
    build_pipeline_by_stage
)
from src.models.transfer_models import build_transfer_model

plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
print("[OK] Preprocessing modules successfully imported with OpenMP safety guards!")


## 2. Load Sample Image for Visual Inspection

Let's select an image displaying moderate-to-severe acne (Level 2 or 3) to clearly observe the effect of each preprocessing step.

In [ ]:
metadata_csv = PROJECT_ROOT / "data" / "metadata" / "acne04_metadata.csv"
df_meta = pd.read_csv(metadata_csv)

# Filter for a representative Level 2 (Moderate) or Level 3 (Severe) image
sample_row = df_meta[df_meta["label"] >= 2].iloc[5]
sample_path = PROJECT_ROOT / sample_row["relative_path"]

original_img = Image.open(sample_path).convert("RGB")
print(f"Selected Sample : {sample_row['filename']}")
print(f"Severity Grade  : {sample_row['severity']}")
print(f"Native Size     : {original_img.size}")

## 3. Step-by-Step Visual Demonstration of All 7 Techniques

In [ ]:
# 1. Stage 1: Raw Resize 224x224
img_stage1 = original_img.resize((224, 224))

# 2. Stage 2: Data Augmentation (simulated preview: flip + rotation)
img_stage2 = img_stage1.transpose(Image.FLIP_LEFT_RIGHT).rotate(10, resample=Image.BILINEAR)

# 3. Stage 3: CLAHE Contrast Enhancement (on LAB Luminance channel)
img_stage3 = apply_clahe(img_stage1, clip_limit=2.5)

# 4. Stage 4: Color Constancy (Gray World Algorithm)
img_stage4 = apply_color_constancy(img_stage3, method="gray_world")

# 5. Stage 5: Face Detection & Cropping
img_stage5 = crop_face(original_img, margin=0.15).resize((224, 224))

# 6. Stage 6: Skin Segmentation & Masking (HSV + YCbCr)
img_stage6 = apply_skin_mask(img_stage5, blend_alpha=0.15)

print("[OK] All image transformations computed successfully!")

### Visual Comparison Across Stages 1 to 6

In [ ]:
stages_preview = [
    ("Stage 1: Raw Baseline (224x224)", img_stage1),
    ("Stage 2: + Augmentation (Flip/Rotate)", img_stage2),
    ("Stage 3: + CLAHE Contrast", img_stage3),
    ("Stage 4: + Color Constancy (WB)", img_stage4),
    ("Stage 5: + Face Detection Crop", img_stage5),
    ("Stage 6: + Skin Masking (HSV/YCbCr)", img_stage6),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (title, img) in enumerate(stages_preview):
    axes[idx].imshow(img)
    axes[idx].set_title(title, fontsize=11, fontweight="bold", pad=8)
    axes[idx].axis("off")

plt.suptitle(f"Incremental Preprocessing Progression\nSample: {sample_row['filename']} ({sample_row['severity']})", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 4. Stage 7: Class Imbalance Solutions (Cost-Sensitive Weights & ROS)

The ACNE04 dataset has a severe class distribution skew (Level 1 has $6.4\times$ more images than Level 3).  
We provide two formal solutions:
1. **Cost-Sensitive CrossEntropy Loss:** Penalizes errors on minority classes inversely to class frequency.
2. **Random Oversampling (ROS):** PyTorch `WeightedRandomSampler` to draw balanced batches.

In [ ]:
train_df = df_meta[df_meta["split"] == "train"]
class_counts = train_df["label"].value_counts().sort_index()

# 1. Compute Cost-Sensitive Class Weights
weights_tensor = compute_class_weights(train_df["label"].values, num_classes=4, method="balanced")

imbalance_summary = pd.DataFrame({
    "Severity Grade": ["Level 0 (Normal)", "Level 1 (Mild)", "Level 2 (Moderate)", "Level 3 (Severe)"],
    "Train Samples": class_counts.values,
    "Sample Proportion (%)": (class_counts.values / len(train_df) * 100).round(2),
    "Computed Loss Weight": weights_tensor.numpy().round(3)
})

display(imbalance_summary)

# 2. Instantiate WeightedRandomSampler
sampler = get_weighted_sampler(train_df["label"].values, num_classes=4)
print(f"\n[OK] WeightedRandomSampler initialized with {sampler.num_samples} draws per epoch.")

## 5. Candidate Architectures Verification (MobileNetV2, EfficientNet-B0, ResNet-50, ViT)

Let us verify that each candidate model can be instantiated and paired with the preprocessing pipeline.

In [ ]:
model_names = ["mobilenet_v2", "efficientnet_b0", "resnet50", "vit"]
dummy_input = torch.randn(2, 3, 224, 224)

print("=== Architecture Verification ===")
for m_name in model_names:
    net = build_transfer_model(model_name=m_name, num_classes=4, pretrained=False)
    net.eval()
    with torch.no_grad():
        out = net(dummy_input)
    param_count = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"  Model: {m_name:16} | Params: {param_count/1e6:6.2f}M | Output Shape: {list(out.shape)}")

## 6. Execution Command Reference for Ablation Experiments

When you are ready to train any model under any specific stage, run the command in your terminal:

```bash
# --- Stage 1: Raw Baseline ---
python src/train.py --model mobilenet_v2 --stage 1 --epochs 30

# --- Stage 2: + Data Augmentation ---
python src/train.py --model mobilenet_v2 --stage 2 --epochs 30

# --- Stage 3: + CLAHE ---
python src/train.py --model mobilenet_v2 --stage 3 --epochs 30

# --- Stage 4: + Color Constancy (White Balance) ---
python src/train.py --model mobilenet_v2 --stage 4 --epochs 30

# --- Stage 5: + Face Detection & Cropping ---
python src/train.py --model mobilenet_v2 --stage 5 --epochs 30

# --- Stage 6: + Skin Masking ---
python src/train.py --model mobilenet_v2 --stage 6 --epochs 30

# --- Stage 7: Full Pipeline (+ Class Imbalance Loss Weights) ---
python src/train.py --model mobilenet_v2 --stage 7 --epochs 30

# To benchmark other architectures (e.g. EfficientNet-B0 or ResNet-50 or ViT):
python src/train.py --model efficientnet_b0 --stage 7 --epochs 30
python src/train.py --model resnet50 --stage 7 --epochs 30
python src/train.py --model vit --stage 7 --epochs 30 --batch_size 16
```